# VoiceGuard — Acoustic Deepfake Classifier Training (Google Colab)

This notebook trains the binary acoustic deepfake classifier on a cloud GPU (e.g. Free-tier Google Colab T4).

### Operating Principles & Honesty Constraints (per `06-DATASETS-AND-TRAINING.md`):
1. **Train in-domain, evaluate out-of-domain**: Report in-domain test EER alongside an unseen-corpus out-of-domain test EER. A large gap is expected and must be reported honestly.
2. **Strict speaker disjointness**: Speakers in the training set must never appear in validation or test splits.
3. **Automatic Mixed Precision (AMP)** and checkpointing enabled to survive transient session resets.

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi

# Install dependencies matching pyproject.toml
!pip install -q timm>=0.9.12 librosa>=0.10.1 soundfile>=0.12.1 scikit-learn>=1.4.0 structlog matplotlib

In [ ]:
# 2. Clone or Link VoiceGuard Repository
import os, sys
from pathlib import Path

# If running in Colab and repository is cloned or uploaded:
repo_root = Path("/content/VoiceGuard").resolve()
if not repo_root.exists():
    repo_root = Path(".").resolve()

backend_dir = repo_root / "backend"
sys.path.insert(0, str(backend_dir))
print(f"Backend loaded from: {backend_dir}")

## 3. Dataset Acquisition (ASVspoof 2019 Logical Access via Kaggle)

Per `06-DATASETS-AND-TRAINING.md` §2.1, the acoustic model trains on the **ASVspoof 2019 Logical Access (LA)** benchmark.

We use **Kaggle** as the primary, high-speed automated source via `kagglehub`.
A synthetic mini-dataset smoke test option is also provided if you want to verify the entire training pipeline in 60 seconds first.

In [ ]:
# ── 3. Dataset Acquisition ─────────────────────────────────────────────
import os, sys
from pathlib import Path

# Set RUN_MINI_TEST = True for an instant 60-second smoke test
# Set RUN_MINI_TEST = False to train on the official ASVspoof 2019 LA dataset from Kaggle
RUN_MINI_TEST = False

data_dir = Path("/content/data")
data_dir.mkdir(parents=True, exist_ok=True)

if RUN_MINI_TEST:
    print("Generating synthetic mini-dataset for instant verification...")
    import numpy as np, soundfile as sf
    mini_dir = data_dir / "mini_asvspoof"
    for split in ["train", "dev"]:
        split_dir = mini_dir / split
        split_dir.mkdir(parents=True, exist_ok=True)
        proto_lines = []
        sr = 16000
        # 60 bonafide + 60 spoof clips per split
        for i in range(120):
            t = np.linspace(0, 3.0, int(3.0 * sr), endpoint=False)
            is_spoof = i >= 60
            spk_id = f"SPK_{split}_{i // 10:02d}"
            key = "spoof" if is_spoof else "bonafide"
            attack = "A01" if is_spoof else "-"
            freq = 220.0 + (50.0 * np.sin(2 * np.pi * 2.0 * t) if not is_spoof else 0.0)
            sig = 0.5 * np.sin(2 * np.pi * freq * t) if not is_spoof else 0.4 * np.sign(np.sin(2 * np.pi * 220.0 * t))
            sig += 0.01 * np.random.randn(len(t))
            fname = f"{split}_{i:04d}.wav"
            sf.write(str(split_dir / fname), sig.astype(np.float32), sr)
            proto_lines.append(f"{spk_id} {fname} - {attack} {key}")
        with open(mini_dir / f"{split}_protocol.txt", "w") as f:
            f.write("\n".join(proto_lines))
    print(f"✓ Created synthetic benchmark at {mini_dir} (120 train, 120 dev audio files)")

else:
    print("Downloading official ASVspoof 2019 LA Dataset via Kaggle...")
    !pip install -q kagglehub
    import kagglehub
    kaggle_path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
    print(f"Kaggle dataset downloaded to: {kaggle_path}")
    
    # Symlink to /content/data/LA for seamless standard path access
    la_src = Path(kaggle_path) / "LA"
    if not la_src.exists():
        # Check if files are directly in kaggle_path or nested
        candidates = list(Path(kaggle_path).rglob("*ASVspoof2019_LA_train*"))
        if candidates:
            la_src = candidates[0].parent
        else:
            la_src = Path(kaggle_path)
    
    target_la = Path("/content/data/LA")
    if not target_la.exists():
        !ln -s "{la_src}" /content/data/LA
    print(f"✓ ASVspoof 2019 LA linked to /content/data/LA")


In [ ]:
# ── 4. Dataset Preprocessing & Speaker Disjointness Verification ───────
# Executes canonicalisation (16 kHz mono 16-bit PCM WAV),
# strictly asserts speaker disjointness across splits,
# and precomputes log-mel spectrogram shards (.npy).

if RUN_MINI_TEST:
    train_proto = "/content/data/mini_asvspoof/train_protocol.txt"
    train_audio = "/content/data/mini_asvspoof/train"
    dev_proto = "/content/data/mini_asvspoof/dev_protocol.txt"
    dev_audio = "/content/data/mini_asvspoof/dev"
else:
    train_proto = "/content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
    train_audio = "/content/data/LA/ASVspoof2019_LA_train/flac"
    dev_proto = "/content/data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
    dev_audio = "/content/data/LA/ASVspoof2019_LA_dev/flac"

print("--- Preprocessing Training Split ---")
!python /content/VoiceGuard/scripts/prepare_datasets.py \
    --protocol {train_proto} \
    --audio-dir {train_audio} \
    --output-dir /content/processed/asvspoof_train \
    --precompute-specs

print("\n--- Preprocessing Development (Validation) Split ---")
!python /content/VoiceGuard/scripts/prepare_datasets.py \
    --protocol {dev_proto} \
    --audio-dir {dev_audio} \
    --output-dir /content/processed/asvspoof_dev \
    --precompute-specs

In [ ]:
# 4. Launch Training Loop
import json
from ai.acoustic.train import run_training

# Load manifests generated by prepare_datasets.py
train_manifest_path = Path("/content/processed/asvspoof_train/manifest.json")
val_manifest_path = Path("/content/processed/asvspoof_dev/manifest.json")

if train_manifest_path.exists() and val_manifest_path.exists():
    with open(train_manifest_path) as f:
        train_manifest = json.load(f)
    with open(val_manifest_path) as f:
        val_manifest = json.load(f)

    training_summary = run_training(
        train_manifest=train_manifest,
        val_manifest=val_manifest,
        output_dir="/content/model_output",
        backbone="efficientnet_b0",
        epochs=30,
        batch_size=32,
        lr=3e-4,
        weight_decay=1e-4,
        patience=6,
    )
    print("Training completed successfully!")
else:
    print("Manifest files not found at specified paths. Ensure data preprocessing is completed.")

In [ ]:
# 5. In-Domain & Out-of-Domain Evaluation
import torch
from ai.acoustic.detector import AcousticDetector
from ai.evaluation.metrics import compute_all_metrics

model_ckpt = Path("/content/model_output/model_best.pt")
if model_ckpt.exists():
    detector = AcousticDetector(
        model_path=model_ckpt,
        backbone="efficientnet_b0",
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    detector.load()
    detector.warmup()
    print("AcousticDetector initialized and warm.")

In [ ]:
# 6. Export Model Artifact
import yaml

artifact_dir = Path("/content/model_output/artifact")
artifact_dir.mkdir(parents=True, exist_ok=True)

config = {
    "name": "voiceguard-acoustic",
    "backbone": "efficientnet_b0",
    "version": "acoustic-efficientnet_b0-v0.1.0",
    "sample_rate": 16000,
    "n_mels": 128,
    "window_seconds": 4.0,
    "hop_seconds": 2.0,
    "top_db": 80.0,
    "aggregation": "trimmed_max_0.65_0.35",
}

with open(artifact_dir / "config.yaml", "w") as f:
    yaml.dump(config, f)

print(f"Artifact exported to {artifact_dir}")